In [1]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
print("Number of GPUs available:", len(tf.config.list_physical_devices('GPU')))
tf.config.list_physical_devices('GPU')

Number of GPUs available: 0


[]

This device doesn't have GPU compatible for tensorflow so it will use CPU by default.

In [3]:
# setting seed for reproducibility
keras.utils.set_random_seed(42)

## Dataset and its description

This dataset is provided by the [Cleveland Clinic Foundation for Heart Disease](https://archive.ics.uci.edu/dataset/45/heart+disease). It's a CSV file with 303 rows. Each row contains information about a patient, and each column describes an attribute of the patient (a feature). We use the features to predict whether a patient has a heart disease (binary classification).

![Description of the dataset](heart_disease.png)

*Figure 1: Description of the dataset*

## Reading the data
This data was provided by [**Francois Chollet**](https://x.com/fchollet) 

In [4]:
df = pd.read_csv("http://storage.googleapis.com/download.tensorflow.org/data/heart.csv")
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,1,145,233,1,2,150,0,2.3,3,0,fixed,0
1,67,1,4,160,286,0,2,108,1,1.5,2,3,normal,1
2,67,1,4,120,229,0,2,129,1,2.6,2,2,reversible,0
3,37,1,3,130,250,0,0,187,0,3.5,3,0,normal,0
4,41,0,2,130,204,0,2,172,0,1.4,1,0,normal,0


In [5]:
df.shape

(303, 14)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    int64  
 1   sex       303 non-null    int64  
 2   cp        303 non-null    int64  
 3   trestbps  303 non-null    int64  
 4   chol      303 non-null    int64  
 5   fbs       303 non-null    int64  
 6   restecg   303 non-null    int64  
 7   thalach   303 non-null    int64  
 8   exang     303 non-null    int64  
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    int64  
 11  ca        303 non-null    int64  
 12  thal      303 non-null    object 
 13  target    303 non-null    int64  
dtypes: float64(1), int64(12), object(1)
memory usage: 33.3+ KB


## Preprocessing of the data

Lets list the categorical and numerical values separately. We can see the datatype of every column in the above description but what we can see from the info() is categorical values are encoded as int64 so lets differentiate the variables manually to categorical and numericals.
This is crucial step, because those categorical column's are encoded numerically, but their values represent categories, not continuous quantities. If we only one-hot encode object columns then only columns with datatype object will be encoded. All the numeric categorical columns such as sex, cp, fbs will remain as number and **TENSORFLOW** will treat the as continuous variable. This measn sex = 1 will be greater that sex =0 which is not correct. SO, manual differentiation of the columns is suggested.

In [7]:
categorical_variables = ['sex', 'cp', 'fbs', 'restecg','exang', 'ca', 'thal']
numerics = ['age', 'trestbps','chol', 'thalach', 'oldpeak', 'slope']

For neural networks and all machine learning algorithms, the inputs are to be  numeric so we will first convert all the categorical data to numeric using one-hot encode and then normalize it.

![pd.get_dummies](pandas.png)

In [8]:
df = pd.get_dummies(df, columns = categorical_variables)
df.head()

,age,trestbps,chol,thalach,oldpeak,slope,target,sex_0,sex_1,cp_0,...,exang_1,ca_0,ca_1,ca_2,ca_3,thal_1,thal_2,thal_fixed,thal_normal,thal_reversible
0,63,145,233,150,2.3,3,0,False,True,False,...,False,True,False,False,False,False,False,True,False,False
1,67,160,286,108,1.5,2,1,False,True,False,...,True,False,False,False,True,False,False,False,True,False
2,67,120,229,129,2.6,2,0,False,True,False,...,True,False,False,True,False,False,False,False,False,True
3,37,130,250,187,3.5,3,0,False,True,False,...,False,True,False,False,False,False,False,False,True,False
4,41,130,204,172,1.4,1,0,True,False,False,...,False,True,False,False,False,False,False,False,True,False
